In [1]:
from keyphrase_curation import *
from keyphrase_curation.controller.annotation import *

alex_controller = AnnotationController("death_penalty", "alexandre")
akira_controller = AnnotationController("death_penalty", "akira")

/home/romario/keyphrase_curation/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
alex_clustering = [1, 26, 10, 12, 12, 26, 10, 9, 10, 16, 10, 20, 16, 12, 17, 24, 17, 14, 12, 17, 8, 16, 24, 15, 13, 5, 13, 5, 12, 28, 20, 9, 14, 5, 20, 5, 13, 8, 8, 24, 28, 8, 28, 2, 24, 21, 2, 28, 25, 18, 28, 11, 11, 7, 2, 11, 7, 32, 11, 18, 1, 3, 4, 1, 4, 10,
                   3, 15, 3, 3, 4, 4, 25, 4, 4, 3, 15, 6, 8, 4, 3, 3, 4, 29, 33, 4, 3, 22, 1, 4, 14, 1, 1, 15, 30, 6, 11, 11, 15, 15, 33, 19, 23, 33, 6, 23, 7, 7, 15, 33, 23, 33, 33, 7, 23, 25, 33, 2, 33, 25, 27, 2, 25, 27, 2, 21, 9, 2, 23, 19, 21, 7, 19, 19, 2, 33, 33, 27, 31, 7]
akira_clustering = [1, 4, 11, 24, 10, 28, 23, 9, 11, 17, 26, 10, 17, 10, 19, 18, 14, 26, 19, 19, 12, 17, 18, 25, 22, 15, 14, 15, 14, 12, 10, 9, 11, 14, 10, 15, 22, 20, 20, 18, 32, 32, 32, 32, 32, 30, 2, 32, 32, 6, 29, 8, 8, 31, 31, 31, 4, 32, 31, 27, 1, 3, 6, 1, 7,
                    23, 3, 16, 33, 3, 7, 3, 5, 3, 7, 3, 16, 21, 20, 7, 3, 3, 7, 3, 13, 7, 3, 21, 1, 7, 11, 1, 1, 16, 24, 21, 8, 16, 16, 16, 33, 33, 5, 6, 33, 4, 33, 33, 33, 33, 4, 28, 33, 33, 13, 33, 1, 2, 33, 33, 33, 2, 33, 33, 2, 30, 9, 2, 10, 33, 6, 33, 33, 33, 2, 33, 5, 33, 10, 33]

In [3]:
len(alex_clustering) == len(akira_clustering)

True

In [4]:
from sklearn.metrics.cluster import fowlkes_mallows_score
from sklearn.metrics.cluster import adjusted_mutual_info_score
from sklearn.metrics.cluster import adjusted_rand_score

def jaccard_dist(ksa, ksb):
    return len(ksa.intersection(ksb)) / len(ksa.union(ksb))


def is_match(ksa, ksb):
    return jaccard_dist(ksa, ksb) > 0.2


def ari(ca, cb):
    return adjusted_rand_score(ca, cb)


def nmi(ca, cb):
    return adjusted_mutual_info_score(ca, cb)


def fms(ca, cb):
    return fowlkes_mallows_score(ca, cb)

print(round(ari(akira_clustering, alex_clustering), 2))
print(round(nmi(akira_clustering, alex_clustering),2))
print(round(fms(akira_clustering, alex_clustering),2))

0.34
0.5
0.38


In [5]:
alex_cs = list(alex_controller.get_cluster_selection().values())[:-1]
akira_cs = list(akira_controller.get_cluster_selection().values())[:-1]

In [6]:
len(alex_cs) == len(akira_cs)

True

In [7]:
def make_data(coder_id, labels):
    data = []
    if len(labels) == 0:
        print("Empty keys")
    for id in range(len(labels)):
        data.append((f'Coder_{coder_id}', id, frozenset([labels[id]])))
    return data


V_ALEX = alex_cs
V_AKIRA = akira_cs

# sklearn kappa
from sklearn.metrics import cohen_kappa_score
print("Kappa: V_A-V_V", cohen_kappa_score(V_ALEX, V_AKIRA))

# krippendorff's alpha
from nltk.metrics.agreement import AnnotationTask
from nltk.metrics.distance import jaccard_distance

va_data = make_data(1, V_ALEX)
vv_data = make_data(2, V_AKIRA)

task = AnnotationTask(data=va_data + vv_data, distance=jaccard_distance)
print("Alpha Jaccard V_A-V_V", round(task.alpha(),2))

Kappa: V_A-V_V 0.375
Alpha Jaccard V_A-V_V 0.38


In [8]:
akira_ks = [0] * len(akira_controller.get_keyphrases_selection())
alex_ks = [0] * len(alex_controller.get_keyphrases_selection())

akira_ks_data = akira_controller.get_keyphrases_selection()
alex_ks_data = alex_controller.get_keyphrases_selection()

for k in akira_ks_data:
    akira_ks[int(k) - 1] = (akira_ks_data[k]["selected1"], akira_ks_data[k]["selected2"])
for k in alex_ks_data:
    alex_ks[int(k) - 1] = (alex_ks_data[k]["selected1"], alex_ks_data[k]["selected2"])


In [9]:
len(akira_ks) == len(alex_ks)

True

In [10]:
for i in range(len(akira_ks)):
    print(akira_ks[i], alex_ks[i])

(6, 24) (11, 0)
(12, 0) (12, 0)
(29, 0) (1, 0)
(22, 0) (22, 0)
(5, 19) (5, 0)
(38, 0) (37, 0)
(26, 28) (26, 0)
(30, 39) (30, 39)
(8, 0) (8, 0)
(67, 82) (67, 0)
(65, 63) (63, 0)
(68, 0) (68, 0)
(86, 0) (86, 0)
(78, 0) (73, 0)
(96, 69) (96, 0)
(91, 0) (33, 0)
(27, 0) (34, 0)
(89, 0) (89, 0)
(83, 0) (83, 0)
(20, 0) (20, 0)
(98, 0) (109, 0)
(115, 0) (113, 0)
(87, 0) (87, 0)
(3, 0) (3, 0)
(21, 0) (21, 0)
(25, 0) (25, 0)
(47, 0) (47, 55)
(41, 42) (48, 0)
(56, 0) (46, 0)
(58, 0) (58, 60)
(45, 0) (45, 0)
(51, 0) (51, 0)
(0, 0) (0, 0)


In [12]:
from nltk.metrics.distance import jaccard_distance
from nltk.metrics.agreement import AnnotationTask


def make_data(coder_id, labels):
    data = []
    if len(labels) == 0:
        print("Empty keys")
    for id in range(len(labels)):
        data.append((f'Coder_{coder_id}', id, frozenset(labels[id])))
    return data


akira_selections = akira_ks
alex_selections = alex_ks

alex_data = make_data(1, alex_selections)
akira_data = make_data(2, akira_selections)

# krippendorff's alpha

task = AnnotationTask(data=alex_data + akira_data, distance=jaccard_distance)
print("Kappa Fleiss Alex-Akira", round(task.multi_kappa(),2))

Kappa Fleiss Alex-Akira 0.63
